# Step 10A — 224px cache, part A - sagittal + coronal

**Settings: Accelerator = `None` (CPU), Internet = `Off`.** Attach the competition data.

## Why two notebooks

At 224px with 32 slices, all three planes would be **21.2 GB** — just over Kaggle's 20 GB per-notebook output limit. So the cache is split: part A holds sagittal and coronal (14.2 GB), part B holds axial (7.1 GB). Training attaches both and concatenates them along the series axis.

**This notebook writes: `cache224_a.npy`, 14.2 GB.**

## What changes from the 160px cache

| | v1 | v2 |
|---|---|---|
| resolution | 160 x 160 | **224 x 224** |
| slices per series | 24 | **32** |

That's 1.4x linear resolution and 33% more through-plane sampling. Both are *new information* rather than tuning — at 160px a meniscus spans roughly 2-3 pixels, which is below what a convolutional network can reliably resolve. At 224px it spans 4-5.

Everything else is byte-identical to the v1 cache: same series selection rule, same geometric slice ordering, same per-series percentile normalisation, same laterality canonicalisation.

Runtime ~28 min on CPU, zero GPU cost.

### 1 · Setup

In [ ]:
import os, glob, time, warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, pydicom
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT  = "/kaggle/working"

PX, NSL = 224, 32
PLANES  = ['Sagittal', 'Coronal']
NSER    = len(PLANES)

train = pd.read_csv(f"{ROOT}/train.csv")
ser   = pd.read_csv(f"{ROOT}/train_series.csv")
studies = train["StudyInstanceUID"].tolist()

print("planes:", PLANES)
print(f"cache ({len(studies)}, {NSER}, {NSL}, {PX}, {PX}) uint8 = "
      f"{len(studies)*NSER*NSL*PX*PX/1e9:.1f} GB   (limit 20 GB)")
print("cores:", os.cpu_count())

### 2 · Series selection — same rule as v1

In [ ]:
def pick(g):
    out = []
    for plane in PLANES:
        sub = g[g["Anatomical_Plane"] == plane]
        if len(sub) == 0:
            out.append(None); continue
        sub = sub.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
        out.append(sub.iloc[0]["SeriesInstanceUID"])
    return out

chosen = {suid: pick(g) for suid, g in ser.groupby("StudyInstanceUID")}
have = np.array([[c is not None for c in chosen.get(s, [None]*NSER)] for s in studies])
for i, p in enumerate(PLANES):
    print(f"{p:<10} present in {have[:, i].mean():.1%} of studies")

### 3 · Decoder — identical to v1 apart from PX and NSL

In [ ]:
def load_series(dirpath, laterality_hint=None):
    files = os.listdir(dirpath)
    if not files:
        return None
    metas = []
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, f), stop_before_pixels=True)
            iop = getattr(d, "ImageOrientationPatient", None)
            ipp = getattr(d, "ImagePositionPatient", None)
            metas.append((f, np.array(iop, float) if iop else None,
                             np.array(ipp, float) if ipp else None,
                             getattr(d, "InstanceNumber", 0),
                             str(getattr(d, "Laterality", "") or "")))
        except Exception:
            continue
    if not metas:
        return None

    iops = [m[1] for m in metas if m[1] is not None]
    if iops:
        iop = iops[0]
        r, c = iop[:3], iop[3:]
        n = np.cross(r, c)
        pos = [(float(np.dot(m[2], n)) if m[2] is not None else float(m[3])) for m in metas]
    else:
        r, c, n = np.array([1.,0,0]), np.array([0,1.,0]), np.array([0,0,1.])
        pos = [float(m[3]) for m in metas]

    order = np.argsort(pos)
    metas = [metas[i] for i in order]
    take  = np.linspace(0, len(metas)-1, NSL).round().astype(int)

    imgs = []
    for i in take:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, metas[i][0]))
            a = d.pixel_array.astype(np.float32)
            if hasattr(d, "RescaleSlope"):
                a = a * float(d.RescaleSlope) + float(getattr(d, "RescaleIntercept", 0))
        except Exception:
            a = np.zeros((PX, PX), np.float32)
        imgs.append(cv2.resize(a, (PX, PX), interpolation=cv2.INTER_AREA))
    vol = np.stack(imgs)

    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo: hi = lo + 1
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    axes = [n, c, r]
    for ax, vec in enumerate(axes):
        dom = int(np.argmax(np.abs(vec)))
        if vec[dom] < 0:
            vol = np.flip(vol, axis=ax)
        axes[ax] = vec * (1 if vec[dom] >= 0 else -1)

    lat = laterality_hint or next((m[4] for m in metas if m[4]), "")
    if str(lat).upper().startswith("L"):
        lr_axis = int(np.argmax([abs(v[0]) for v in axes]))
        vol = np.flip(vol, axis=lr_axis)

    return (vol * 255).astype(np.uint8)

def study_paths(suid):
    return [(f"{ROOT}/train_series/{suid}/{c}" if c else None) for c in chosen.get(suid, [None]*NSER)]

print("decoder ready")

### 4 · Build

Resumable via `done224_a.npy` — a crash costs minutes, not the run.

In [ ]:
CACHE = f"{OUT}/cache224_a.npy"
DONE  = f"{OUT}/done224_a.npy"
SHAPE = (len(studies), NSER, NSL, PX, PX)

if not os.path.exists(CACHE):
    np.lib.format.open_memmap(CACHE, mode="w+", dtype=np.uint8, shape=SHAPE)
    np.save(DONE, np.zeros(len(studies), np.uint8))
    print("created", CACHE)

done = np.load(DONE)
print(f"already done: {done.sum():,}/{len(studies):,}")

def work(i):
    a = np.lib.format.open_memmap(CACHE, mode="r+")
    got = 0
    for k, p in enumerate(study_paths(studies[i])):
        if p and os.path.isdir(p):
            try:
                v = load_series(p)
                if v is not None:
                    a[i, k] = v; got += 1
            except Exception:
                pass
    del a
    return i, got

todo = [i for i in range(len(studies)) if not done[i]]
print(f"to process: {len(todo):,}\n")

t0 = time.time()
with Pool(4) as pool:
    for n, (i, got) in enumerate(pool.imap_unordered(work, todo, chunksize=8), 1):
        done[i] = 1
        if n % 200 == 0 or n == len(todo):
            np.save(DONE, done)
            el = time.time() - t0
            print(f"  {n}/{len(todo)}  {el/n:.2f}s/study  eta {(len(todo)-n)*el/n/60:.0f} min", flush=True)
np.save(DONE, done)
print(f"\ncomplete: {done.sum():,}/{len(studies):,} in {(time.time()-t0)/60:.0f} min")

### 5 · Verify

In [ ]:
import matplotlib.pyplot as plt
arr = np.load(CACHE, mmap_mode="r")
rng = np.random.default_rng(0)
chk = rng.choice(len(studies), 300, replace=False)

nz = np.array([[arr[i, k].max() > 0 for k in range(NSER)] for i in chk])
print("shape", arr.shape)
for i, p in enumerate(PLANES):
    print(f"  {p:<10} populated {nz[:, i].mean():.1%}")
print(f"studies with >=1 series: {nz.any(1).mean():.1%}")
print(f"intensity mean {arr[chk[:40]].mean():.1f} std {arr[chk[:40]].std():.1f}")

i = int(np.where(done > 0)[0][0])
fig, ax = plt.subplots(NSER, 8, figsize=(18, 2.6*NSER), squeeze=False)
for k in range(NSER):
    for j in range(8):
        ax[k, j].imshow(arr[i, k, j*4], cmap="gray", vmin=0, vmax=255)
        ax[k, j].axis("off")
    ax[k, 0].set_title(PLANES[k], loc="left", fontsize=11)
plt.suptitle("224px cache - every 4th slice")
plt.tight_layout(); plt.show()